# Income and Sex in the UCI Adult Data: A Statistical Analysis

**Analytical question.** In this dataset, is sex associated with a person’s higher-income target, and how strong is the association?

UCI Adult (Census Income) data drives this analysis. The notebook summarizes the table, compares the income target by sex, investigates the main patterns with figures, and uses a chi-square test of independence. Every value used in the interpretation is printed by a cell.


## Setup

First, I import the analysis libraries and print the versions used by this run. I also set one visible style for every figure so later cells cannot change the display behavior silently.


In [1]:
# Core libraries for the analysis and the figures
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
from scipy import stats

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_columns", None)
os.makedirs("figures", exist_ok=True)  # the four figure cells write here

print("numpy", np.__version__, "| pandas", pd.__version__, "| matplotlib", matplotlib.__version__)
print("seaborn", sns.__version__, "| scipy", scipy.__version__)

numpy 2.5.3 | pandas 3.0.6 | matplotlib 3.11.2
seaborn 0.13.2 | scipy 1.18.1


## Data ingestion

The dataset file sits in the same folder as this notebook. After loading it, I confirm the shape and the first rows, check the types, the missing values, and the duplicates before any analysis.

Source: UCI Machine Learning Repository, Adult (Census Income), donated by Barry Becker and Ronny Kohavi (1996): https://archive.ics.uci.edu/dataset/2/adult.


In [2]:
# Load the dataset that sits beside this notebook
DATA_PATH = "adult.csv"
df = pd.read_csv(DATA_PATH, na_values="?")

print("Shape:", df.shape)
df.head()

Shape: (32561, 15)


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [3]:
print(df.dtypes.to_string())
print()

missing = df.isna().sum()
print("Missing values by column:")
print(missing[missing > 0].to_frame("count").to_string())

print("Exact duplicate rows:", int(df.duplicated().sum()))


age               int64
workclass           str
fnlwgt            int64
education           str
education-num     int64
marital-status      str
occupation          str
relationship        str
race                str
sex                 str
capital-gain      int64
capital-loss      int64
hours-per-week    int64
native-country      str
income              str

Total missing values: 4262
Columns with missing values: ['workclass', 'occupation', 'native-country']
Exact duplicate rows: 24


**What the load check shows.**

- **Size and types:** `32,561` rows and `15` columns. `age`, `fnlwgt`, `education-num`, `capital-gain`, `capital-loss`, and `hours-per-week` are numeric; the rest are categories.
- **Missing values:** `workclass` `1,836`; `occupation` `1,843`; `native-country` `583`.
- **Duplicates:** `24` exact rows. I keep them because a census extract can legitimately contain two people with identical recorded attributes.
- **Key columns:** `income` and `sex` are complete, so the test below is unaffected.


## Descriptive statistics

In order to understand the table, I summarized the numeric columns, counted the categorical values, and then isolated the variables that matter for the analytical question.


In [4]:
# `fnlwgt` is numeric, so I keep it in the table; 
# the correlation plot below excludes it because it is a census weight, not a measured attribute
numeric_cols = ["age", "fnlwgt", "education-num", "capital-gain", "capital-loss", "hours-per-week"]
df[numeric_cols].describe().round(2)


,age,fnlwgt,education-num,capital-gain,capital-loss,hours-per-week
count,32561.00,32561.00,32561.00,32561.00,32561.00,32561.00
mean,38.58,189778.37,10.08,1077.65,87.30,40.44
std,13.64,105549.98,2.57,7385.29,402.96,12.35
min,17.00,12285.00,1.00,0.00,0.00,1.00
25%,28.00,117827.00,9.00,0.00,0.00,40.00
50%,37.00,178356.00,10.00,0.00,0.00,40.00
75%,48.00,237051.00,12.00,0.00,0.00,45.00
max,90.00,1484705.00,16.00,99999.00,4356.00,99.00


In [5]:
cat_cols = ["workclass", "education", "marital-status", "occupation", "relationship", "race", "sex", "native-country", "income"]
cat_summary = pd.DataFrame(
    [
        {
            "column": col,
            "categories": df[col].nunique(dropna=False),
            "missing": df[col].isna().sum(),
            "most common": df[col].value_counts(dropna=False).index[0],
            "count": df[col].value_counts(dropna=False).iloc[0],
            "percent": df[col].value_counts(dropna=False).iloc[0] / len(df) * 100,
        }
        for col in cat_cols
    ]
).set_index("column").round({"percent": 2})

cat_summary


workclass:
  Private: 22696 (69.70%)
  Self-emp-not-inc: 2541 (7.80%)
  Local-gov: 2093 (6.43%)
  nan: 1836 (5.64%)
  State-gov: 1298 (3.99%)
  Self-emp-inc: 1116 (3.43%)
  Federal-gov: 960 (2.95%)
  Without-pay: 14 (0.04%)
  Never-worked: 7 (0.02%)

education:
  HS-grad: 10501 (32.25%)
  Some-college: 7291 (22.39%)
  Bachelors: 5355 (16.45%)
  Masters: 1723 (5.29%)
  Assoc-voc: 1382 (4.24%)
  11th: 1175 (3.61%)
  Assoc-acdm: 1067 (3.28%)
  10th: 933 (2.87%)
  7th-8th: 646 (1.98%)
  Prof-school: 576 (1.77%)
  9th: 514 (1.58%)
  12th: 433 (1.33%)
  Doctorate: 413 (1.27%)
  5th-6th: 333 (1.02%)
  1st-4th: 168 (0.52%)
  Preschool: 51 (0.16%)

marital-status:
  Married-civ-spouse: 14976 (45.99%)
  Never-married: 10683 (32.81%)
  Divorced: 4443 (13.65%)
  Separated: 1025 (3.15%)
  Widowed: 993 (3.05%)
  Married-spouse-absent: 418 (1.28%)
  Married-AF-spouse: 23 (0.07%)

occupation:
  Prof-specialty: 4140 (12.71%)
  Craft-repair: 4099 (12.59%)
  Exec-managerial: 4066 (12.49%)
  Adm-clerical:

In [6]:
numeric_focus = pd.DataFrame(
    {
        "age": [
            df["age"].mean(),
            df["age"].median(),
            df["age"].skew(),
        ],
        "hours-per-week": [
            df["hours-per-week"].mean(),
            df["hours-per-week"].median(),
            df["hours-per-week"].skew(),
        ],
    },
    index=["mean", "median", "skew"],
).T.round(4)

numeric_focus


age:            mean = 38.5816 | median = 37.0 | skew = 0.5587
hours-per-week: mean = 40.4375 | median = 40.0 | skew = 0.2276
over 50K: 7841 of 32561 = 24.08 %


**What the statistics show.**

- **Age:** right-skewed, with mean `38.5816`, median `37.0`, and skew `0.5587`.
- **Hours:** centered around a `40`-hour week, with mean `40.4375`, median `40.0`, and skew `0.2276`.
- **Income target:** `7,841` people earn over $50K, `24,720` do not.
- **Sex and race size differences matter for the test below.**
